# Phase 3 Architecture A v1

Based on Phase 2 v5 architecture with two key changes:

1. **Fixed D1 ↔ Intent labels** — 574 misaligned intent labels corrected (trust D1, derive intent)
2. **BGE-base-en-v1.5 embeddings** — upgraded from MiniLM-L6 (384d) to BGE-base (768d) for stronger semantic discrimination at tier boundaries
3. **PCA = 60** — increased from 40 to capture richer BGE embedding space

Everything else (50 handcrafted features, two-stage D-score conditioning, blended tier, capped sample weights, OOF tier features) is identical to Phase 2 v5.

In [1]:
# Colab setup
!pip install -q xgboost sentence-transformers scikit-learn

In [2]:
import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5
PCA_COMPONENTS = 60  # Phase 3: increased from 40 to 60 for 768d BGE embeddings

## Load Dataset

Phase 3 dataset: D1↔intent labels already fixed (574 rows corrected).

In [3]:
DATA_PATH = '/content/prompt_classifier_phase3_v1_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(2421, 29)


,id,prompt,intent,task_type,reasoning_chain_detected,d1,d2,d3,d4,d5,...,prompting_techniques,prompt_type,phrasing_style,domain,source,augmentation_group,intent_d1_mismatch_flag,boundary_t12_flag,boundary_t23_flag,original_row_flag
0,NaN,Imagine you are a science fiction author renow...,SYNTHETIC,generation,True,0.75,0.50,0.50,0.0,0.0,...,"['ROLE_PROMPTING', 'TREE_OF_THOUGHTS']",CREATIVE_WRITING,NaN,NaN,phase2,NaN,False,False,False,True
1,NaN,You are an expert photography tutor. I want to...,ANALYTICAL,generation,True,0.50,0.75,0.50,0.0,0.0,...,['CODE_PROMPTING'],CODE_EXPLANATION,NaN,NaN,phase2,NaN,False,True,False,True
2,NaN,You are a leading neuroscientist specializing ...,SYNTHETIC,reasoning,True,0.75,0.75,0.75,0.5,0.5,...,"['ROLE_PROMPTING', 'CHAIN_OF_THOUGHT']",CONVERSATIONAL,NaN,NaN,phase2,NaN,True,False,True,True
3,NaN,I want to understand the basic human emotions....,FACTUAL,generation,False,0.00,0.50,0.50,0.0,0.0,...,"['CHAIN_OF_THOUGHT', 'CONTEXTUAL_PROMPTING']",COMPARISON,NaN,NaN,phase2,NaN,False,False,False,True
4,NaN,Here are examples of competitive exclusion. Ex...,SYNTHETIC,reasoning,True,0.75,0.50,0.50,0.0,0.5,...,['ONE_SHOT_FEW_SHOT'],PROGRAMMING_CODE_GENERATION,NaN,NaN,phase2,NaN,True,False,False,True


## Dataset Safety Fixes and Validation

Defensive checks carried forward from v5. The intent fix is already applied in the CSV, but formatting merge and boolean normalization run defensively.

In [4]:
SCORE_COLS = ['d1', 'd2', 'd3', 'd4', 'd5']
VALID_SCORES = [0.0, 0.25, 0.5, 0.75, 1.0]
SCORE_TO_CLASS = {score: idx for idx, score in enumerate(VALID_SCORES)}
CLASS_TO_SCORE = {idx: score for score, idx in SCORE_TO_CLASS.items()}

DIMENSION_LABELS = {
    'd1': 'Semantic Complexity',
    'd2': 'Domain Specificity',
    'd3': 'Output Formality',
    'd4': 'Research Dependency',
    'd5': 'Context Requirement',
}

# --- Correct D1 -> Intent mapping (ground truth) ---
D1_TO_INTENT = {
    0.00: 'FACTUAL',
    0.25: 'FACTUAL',
    0.50: 'ANALYTICAL',
    0.75: 'SYNTHETIC',
    1.00: 'STRATEGIC',
}


def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() == 'true'


def has_any_term(text, terms):
    for term in terms:
        if re.search(rf'(?<![A-Za-z0-9_]){re.escape(term)}(?![A-Za-z0-9_])', text):
            return True
    return False


def rederive_question_task_type(prompt):
    lower = str(prompt).lower()
    if has_any_term(lower, ['python', 'sql query', 'source code', 'code', 'function', 'script', 'debug', 'yaml', 'json', 'syntax error', 'stack trace', 'kubernetes manifest']):
        return 'coding'
    if has_any_term(lower, ['summarize', 'summary', 'tl;dr', 'condense']):
        return 'summarisation'
    if has_any_term(lower, ['create', 'draft', 'write', 'generate', 'compose', 'build']):
        return 'generation'
    return 'reasoning'


def complexity_score_from_dims_frame(frame):
    return (
        frame['d1'] * 0.35 +
        frame['d2'] * 0.20 +
        frame['d3'] * 0.20 +
        frame['d4'] * 0.15 +
        frame['d5'] * 0.10
    )


def tier_from_score(score):
    if score < 0.40:
        return 'T1'
    if score < 0.70:
        return 'T2'
    return 'T3'


def parse_research_signals(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(value)
    except Exception:
        try:
            parsed = ast.literal_eval(str(value))
        except Exception:
            return []
    return parsed if isinstance(parsed, list) else []


# --- Defensive label cleanup ---

# Verify D1 <-> Intent alignment (should be 0 after fix)
df['expected_intent'] = df['d1'].map(D1_TO_INTENT)
misaligned_count = int((df['intent'] != df['expected_intent']).sum())
if misaligned_count > 0:
    print(f'WARNING: {misaligned_count} D1<->intent misalignments found, fixing...')
    df['intent'] = df['expected_intent']
else:
    print(f'D1<->intent alignment check: OK (0 misalignments)')
df.drop(columns=['expected_intent'], inplace=True)

# Question classification fix
question_classification_mask = (
    (df.get('source', '') == 'phase1') &
    df['prompt'].astype(str).str.strip().str.endswith('?') &
    (df['task_type'] == 'classification')
)
fixed_question_count = int(question_classification_mask.sum())
if fixed_question_count:
    df.loc[question_classification_mask, 'task_type'] = df.loc[question_classification_mask, 'prompt'].apply(rederive_question_task_type)

# Merge formatting into generation (too few samples)
formatting_count = int((df['task_type'] == 'formatting').sum())
df.loc[df['task_type'] == 'formatting', 'task_type'] = 'generation'

# Merge sparql_generation into generation (too few samples)
sparql_count = int((df['task_type'] == 'sparql_generation').sum())
df.loc[df['task_type'] == 'sparql_generation', 'task_type'] = 'generation'

df['reasoning_chain_detected'] = df['reasoning_chain_detected'].apply(normalize_bool)
df['research_signals_parsed'] = df['research_signals'].apply(parse_research_signals)
df['computed_complexity_score'] = complexity_score_from_dims_frame(df)
df['computed_tier'] = df['computed_complexity_score'].apply(tier_from_score)

# Remove any duplicates
dup_count = int(df['prompt'].duplicated().sum())
if dup_count:
    df = df.drop_duplicates(subset='prompt', keep='first').reset_index(drop=True)

print(f'Question task_type fixes applied: {fixed_question_count}')
print(f'Formatting rows merged into generation: {formatting_count}')
print(f'sparql_generation rows merged into generation: {sparql_count}')
print(f'Tier formula mismatches: {(df["computed_tier"] != df["tier"]).sum()}')
print(f'Max complexity score deviation: {np.abs(df["computed_complexity_score"] - df["complexity_score"]).max()}')
print(f'Duplicate prompts removed: {dup_count}')
print(f'\nFinal dataset: {len(df)} rows')

print(f'\nTier counts:')
print(df['tier'].value_counts().sort_index())
print(f'\nIntent counts:')
print(df['intent'].value_counts())
print(f'\nTask type counts:')
print(df['task_type'].value_counts())

D1<->intent alignment check: OK (0 misalignments)
Question task_type fixes applied: 0
Formatting rows merged into generation: 10
sparql_generation rows merged into generation: 8
Tier formula mismatches: 0
Max complexity score deviation: 1.1102230246251565e-16
Duplicate prompts removed: 0

Final dataset: 2421 rows

Tier counts:
tier
T1     970
T2    1072
T3     379
Name: count, dtype: int64

Intent counts:
intent
SYNTHETIC     856
ANALYTICAL    791
FACTUAL       577
STRATEGIC     197
Name: count, dtype: int64

Task type counts:
task_type
reasoning         1314
generation         800
summarisation      120
classification     100
coding              87
Name: count, dtype: int64


## 50 Hand-Crafted Features

Identical to Phase 2 v5: 32 Phase 1 features + 10 Phase 2 features + 8 v5 D1/D2-specific features.

In [5]:
ARTIFACT_TERMS = ['csv', 'json', 'pdf', 'log', 'yaml', 'yml', 'xlsx', 'docx', 'transcript', 'diagram']
CLOUD_PROVIDERS = ['aws', 'azure', 'gcp', 'google cloud', 'oci']
SYSTEMS = ['salesforce', 'servicenow', 'jira', 'workday', 'sap', 'snowflake', 'databricks', 'okta', 'hubspot', 'github', 'gitlab']
FRAMEWORKS = ['itil', 'finops', 'togaf', 'owasp', 'dora', 'nist', 'hipaa', 'soc 2', 'soc2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss']
VENDOR_TOOLS = sorted(set(CLOUD_PROVIDERS + SYSTEMS + ['openai', 'anthropic', 'bedrock', 'terraform', 'kubernetes', 'docker', 'jenkins', 'splunk']))

DOMAIN_BUCKETS = {
    'cloud': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'budget', 'chargeback', 'showback'],
    'security': ['security', 'vulnerability', 'iam', 'zero trust', 'soc'],
    'devops': ['devops', 'ci/cd', 'pipeline', 'sre', 'deployment'],
    'data': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai': ['ai', 'llm', 'genai', 'machine learning', 'model'],
    'hr': ['hr', 'employee', 'talent', 'workforce', 'recruiting'],
    'supply': ['supply chain', 'inventory', 'procurement', 'logistics'],
}

D1_COMPLEXITY_TERMS = [
    'strategic', 'cross-domain', 'enterprise-wide', 'synthesize', 'multi-cloud',
    'governance', 'architecture', 'framework', 'transformation', 'lifecycle',
    'holistic', 'end-to-end', 'migration', 'orchestration',
]

SIMPLE_FACTUAL_PATTERNS = [
    r'^what is\b', r'^define\b', r'^who is\b', r'^when\b',
    r'^list\b', r'^name\b', r'\bwhat does .{1,30} mean\b',
]

D2_DOMAIN_TERMS = [
    'kubernetes', 'terraform', 'sagemaker', 'databricks', 'snowflake',
    'cicd', 'ci/cd', 'finops', 'mlops', 'devsecops', 'apigee',
    'oauth', 'saml', 'oidc', 'vpc', 'subnet', 'iam',
]


def count_terms_safe(text, terms):
    return sum(1 for term in terms if term in text)


def any_terms_safe(text, terms):
    return any(term in text for term in terms)


def get_style_at(phrasing_styles, i):
    if phrasing_styles is None:
        return None
    try:
        value = phrasing_styles.iloc[i]
    except AttributeError:
        value = phrasing_styles[i]
    return None if pd.isna(value) else str(value).strip().lower()


def handcrafted_features(prompts, phrasing_styles=None):
    rows = []
    for i, prompt in enumerate(prompts):
        text = str(prompt)
        lower = text.lower()
        words = re.findall(r'\b\w+\b', lower)
        unique_words = set(words)
        sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
        lines = [line for line in text.splitlines() if line.strip()]
        style = get_style_at(phrasing_styles, i)

        row = {}

        # Text statistics: 6
        row['char_len'] = len(text)
        row['word_count'] = len(words)
        row['sentence_count'] = max(1, len(sentences))
        row['avg_word_len'] = float(np.mean([len(w) for w in words])) if words else 0.0
        row['unique_word_ratio'] = len(unique_words) / max(1, len(words))
        row['line_count'] = len(lines)

        # D5 Context Requirement: 4
        row['has_attachment'] = int(any_terms_safe(lower, ['uploaded', 'attached', 'provided file', 'document below', 'context below', 'see below']))
        row['provided_artifact_count'] = count_terms_safe(lower, ARTIFACT_TERMS)
        row['large_context_signal'] = int(any_terms_safe(lower, ['across all', 'entire', 'all of our', 'company-wide', 'large context', 'full document']))
        row['multi_document_signal'] = int(any_terms_safe(lower, ['multiple', 'all the', 'each of the', 'various', 'several documents', 'set of files']))

        # D3 Output Formality: 4
        row['has_formal_deliverable'] = int(any_terms_safe(lower, ['report', 'brief', 'proposal', 'specification', 'whitepaper', 'requirements doc']))
        row['has_report_package'] = int(any_terms_safe(lower, ['appendix', 'table of contents', 'risk register', 'executive summary', 'roadmap', 'implementation plan']))
        row['has_long_output_signal'] = int(any_terms_safe(lower, ['comprehensive', 'detailed', 'thorough', 'in-depth', 'end-to-end']))
        row['structured_section_count'] = count_terms_safe(lower, ['executive summary', 'timeline', 'roadmap', 'risk register', 'assumptions', 'recommendations', 'next steps', 'success metrics'])

        # D1 Semantic Complexity: 3
        row['has_scope_words'] = int(any_terms_safe(lower, ['strategic', 'cross-domain', 'enterprise-wide', 'synthesize', 'multi-cloud', 'governance']))
        row['action_verb_count'] = count_terms_safe(lower, ['build', 'design', 'evaluate', 'integrate', 'optimize', 'develop', 'assess', 'recommend', 'compare'])
        row['multi_stage_signal'] = int(bool(re.search(r'\bphase\b|\bstage\b|\bstep\s*1\b|\bmilestone\b|\bsequentially\b|\bfirst\b.*\bthen\b', lower)))

        # D2 Domain Specificity: 4
        row['has_compliance'] = int(any_terms_safe(lower, ['nist', 'hipaa', 'soc2', 'soc 2', 'gdpr', 'iso 27001', 'pci-dss', 'pci dss', 'compliance']))
        row['cloud_providers_mentioned'] = count_terms_safe(lower, CLOUD_PROVIDERS)
        row['systems_mentioned'] = count_terms_safe(lower, SYSTEMS)
        row['domain_framework_count'] = count_terms_safe(lower, FRAMEWORKS)

        # D4 Research Dependency: 5
        row['external_data_score'] = count_terms_safe(lower, ['market research', 'industry report', 'analyst', 'third-party', 'external data', 'latest', 'current'])
        row['has_time_reference'] = int(bool(re.search(r'\b20\d{2}\b|\bfy\d{2}\b|\bthis quarter\b|\blatest\b|\bcurrent\b|\brecent\b|\btoday\b|\bnow\b', lower)))
        row['vendor_tool_count'] = count_terms_safe(lower, VENDOR_TOOLS)
        row['has_market_terms'] = int(any_terms_safe(lower, ['competitor', 'market share', 'tam', 'sam', 'som', 'benchmark', 'industry trend']))
        row['has_cost_comparison'] = int(any_terms_safe(lower, ['pricing', 'tco', 'roi', 'showback', 'chargeback', 'cheapest']) or 'cost analysis' in lower)

        # Boundary/Risk: 3
        row['has_comparison'] = int(any_terms_safe(lower, ['compare', 'versus', 'tradeoff']) or any(phrase in lower for phrase in [' vs ', 'difference between']))
        row['stakeholder_mentions'] = count_terms_safe(lower, ['ceo', 'cto', 'cio', 'cfo', 'board', 'leadership', 'management', 'executive'])
        row['risk_language'] = count_terms_safe(lower, ['risk', 'threat', 'vulnerability', 'mitigation', 'breach', 'exposure', 'audit'])

        # Phase 2 Intent / Reasoning Chain: 5
        row['has_role_prompt'] = int(bool(re.search(r'\byou are\b|\bact as\b|\bassume the role\b', lower)))
        row['has_step_request'] = int(bool(re.search(r'\bstep[- ]by[- ]step\b|\bfirst\b.*\bthen\b|\bsequentially\b', lower)))
        row['has_chain_of_thought'] = int(bool(re.search(r'\bthink through\b|\breason about\b|\blet.s think\b|\bchain of thought\b|\bwalk me through\b', lower)))
        if '?' not in text:
            row['question_complexity'] = 0
        elif any(phrase in lower for phrase in ['what should', 'design a']) or any_terms_safe(lower, ['recommend', 'propose', 'strategy']):
            row['question_complexity'] = 3
        elif any_terms_safe(lower, ['why', 'how', 'compare', 'analyze', 'evaluate', 'assess']):
            row['question_complexity'] = 2
        else:
            row['question_complexity'] = 1
        row['multi_domain_count'] = sum(1 for bucket_terms in DOMAIN_BUCKETS.values() if any_terms_safe(lower, bucket_terms))

        # Phase 2 Task Type: 5
        row['has_code_block'] = int('```' in text)
        row['has_output_format'] = int(bool(re.search(r'\bin json\b|\bas a table\b|\bformat as\b|\bcsv output\b|\bin yaml\b|\bas markdown\b|\bstrict yaml\b|\bstrict json\b', lower)))
        row['has_creative_language'] = int(any_terms_safe(lower, ['imagine', 'creative', 'story', 'compose', 'fictional']) or 'write a' in lower)
        row['has_classification_request'] = int(any_terms_safe(lower, ['classify', 'categorize', 'label']) or any(phrase in lower for phrase in ['which category', 'sort into']))
        row['enumeration_signal'] = int(bool(re.search(r'\blist\b|\btop \d+\b|\benumerate\b|\bbullet point\b|\brank\b', lower)))

        # v5 D1/D2-specific signals: 8
        row['d1_strategic_signal_count'] = count_terms_safe(lower, D1_COMPLEXITY_TERMS)
        row['d1_simple_factual_signal'] = int(any(re.search(pattern, lower) for pattern in SIMPLE_FACTUAL_PATTERNS))
        row['d1_multi_constraint_count'] = count_terms_safe(lower, ['include', 'cover', 'consider', 'account for', 'must'])
        row['d1_solution_design_signal'] = int(any_terms_safe(lower, ['design', 'architect', 'plan', 'strategy', 'roadmap']))
        row['d2_domain_term_count'] = count_terms_safe(lower, D2_DOMAIN_TERMS)
        row['d2_acronym_count'] = len(re.findall(r'\b[A-Z]{2,6}\b', text))
        row['d2_vendor_or_framework_signal'] = int(row['vendor_tool_count'] > 0 or row['domain_framework_count'] > 0)
        row['d2_generic_prompt_signal'] = int(row['d2_domain_term_count'] == 0 and row['cloud_providers_mentioned'] == 0 and row['systems_mentioned'] == 0)

        # Phrasing style: 3
        row['phrasing_explicit'] = int(style == 'explicit')
        row['phrasing_implicit'] = int(style == 'implicit')
        row['phrasing_vague'] = int(style == 'vague')

        rows.append(row)

    feature_df = pd.DataFrame(rows).fillna(0)
    expected_features = 50
    if feature_df.shape[1] != expected_features:
        raise ValueError(f'Expected {expected_features} hand-crafted features, got {feature_df.shape[1]}')
    return feature_df

hand_df = handcrafted_features(df['prompt'], df.get('phrasing_style'))
print('Hand-crafted feature shape:', hand_df.shape)
hand_df.head()

Hand-crafted feature shape: (2421, 50)


,char_len,word_count,sentence_count,avg_word_len,unique_word_ratio,line_count,has_attachment,provided_artifact_count,large_context_signal,multi_document_signal,...,d1_simple_factual_signal,d1_multi_constraint_count,d1_solution_design_signal,d2_domain_term_count,d2_acronym_count,d2_vendor_or_framework_signal,d2_generic_prompt_signal,phrasing_explicit,phrasing_implicit,phrasing_vague
0,1058,158,11,5.506329,0.696203,4,0,1,0,0,...,0,2,0,0,0,1,0,0,0,0
1,531,82,9,5.146341,0.707317,1,0,1,0,0,...,0,1,0,0,1,0,1,0,0,0
2,525,70,5,6.371429,0.814286,1,0,1,0,0,...,0,2,0,0,0,0,1,0,0,0
3,237,38,4,5.078947,0.921053,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,713,107,12,5.364486,0.644860,1,0,1,0,0,...,0,1,1,0,0,0,1,0,0,0


## Embeddings — BGE-base-en-v1.5

**Phase 3 change:** Upgraded from `all-MiniLM-L6-v2` (384d, 22M params) to `BAAI/bge-base-en-v1.5` (768d, 109M params).

BGE-base provides significantly stronger semantic discrimination at tier boundaries. The model uses a query prefix for best performance.

In [6]:
# Phase 3: BGE-base-en-v1.5 (768d) — upgraded from MiniLM-L6 (384d)
EMBEDDING_MODEL_NAME = 'BAAI/bge-base-en-v1.5'
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

prompts = df['prompt'].astype(str).tolist()

# BGE models perform best with a query prefix
prompts_for_encoding = ['Represent this sentence: ' + p for p in prompts]

embeddings = embedding_model.encode(
    prompts_for_encoding,
    batch_size=32,  # smaller batch for larger model
    show_progress_bar=True,
    normalize_embeddings=True,
)

print('Embedding model:', EMBEDDING_MODEL_NAME)
print('Embedding shape:', embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Embedding model: BAAI/bge-base-en-v1.5
Embedding shape: (2421, 768)


## Feature Matrix Helpers

In [7]:
def fit_shared_transformers(train_embeddings, train_hand_features):
    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    train_emb_pca = pca.fit_transform(train_embeddings)
    train_raw = np.hstack([train_emb_pca, train_hand_features])

    scaler = StandardScaler()
    train_X = scaler.fit_transform(train_raw)
    return pca, scaler, train_X


def transform_shared_features(embeddings_part, hand_features_part, pca, scaler):
    emb_pca = pca.transform(embeddings_part)
    raw = np.hstack([emb_pca, hand_features_part])
    return scaler.transform(raw)


def fit_full_feature_matrix():
    pca, scaler, X_full = fit_shared_transformers(embeddings, hand_df.values)
    print('PCA variance explained:', round(float(pca.explained_variance_ratio_.sum()), 4))
    print('Base feature shape:', X_full.shape)  # Should be (N, 110) = 60 PCA + 50 handcrafted
    return pca, scaler, X_full

## Label Encoding

In [8]:
label_encoders = {}
targets = {}

for col in SCORE_COLS:
    unknown_scores = sorted(set(df[col].dropna().astype(float)) - set(VALID_SCORES))
    if unknown_scores:
        raise ValueError(f'{col} has invalid scores: {unknown_scores}')
    targets[col] = df[col].astype(float).map(SCORE_TO_CLASS).astype(int).values

for col in ['tier', 'intent', 'task_type']:
    le = LabelEncoder()
    targets[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

targets['reasoning_chain_detected'] = df['reasoning_chain_detected'].astype(bool).astype(int).values

head_classes = {
    'tier': len(label_encoders['tier'].classes_),
    'intent': len(label_encoders['intent'].classes_),
    'task_type': len(label_encoders['task_type'].classes_),
    'reasoning_chain_detected': 2,
    'd1': 5,
    'd2': 5,
    'd3': 5,
    'd4': 5,
    'd5': 5,
}

STAGE1_HEADS = ['tier', 'intent', 'task_type', 'reasoning_chain_detected']
ALL_HEADS = STAGE1_HEADS + SCORE_COLS
print('Head classes:', head_classes)

tier: ['T1', 'T2', 'T3']
intent: ['ANALYTICAL', 'FACTUAL', 'STRATEGIC', 'SYNTHETIC']
task_type: ['classification', 'coding', 'generation', 'reasoning', 'summarisation']
Head classes: {'tier': 3, 'intent': 4, 'task_type': 5, 'reasoning_chain_detected': 2, 'd1': 5, 'd2': 5, 'd3': 5, 'd4': 5, 'd5': 5}


## Model Helpers

In [9]:
def make_xgb(num_classes, seed=RANDOM_STATE):
    is_binary = num_classes == 2
    params = dict(
        n_estimators=350,
        max_depth=3,
        learning_rate=0.045,
        subsample=0.88,
        colsample_bytree=0.88,
        reg_lambda=2.5,
        min_child_weight=5,
        random_state=seed,
        eval_metric='logloss' if is_binary else 'mlogloss',
        objective='binary:logistic' if is_binary else 'multi:softprob',
        tree_method='hist',
    )
    if not is_binary:
        params['num_class'] = num_classes
    return XGBClassifier(**params)


def capped_sample_weight(y, cap=3.0):
    weights = compute_sample_weight(class_weight='balanced', y=y)
    return np.clip(weights, 1.0 / cap, cap)


WEIGHT_CAPS = {
    'tier': 2.2,
    'intent': 3.0,
    'task_type': 2.5,
    'reasoning_chain_detected': 2.0,
    'd1': 4.0,
    'd2': 4.0,
    'd3': 3.0,
    'd4': 3.0,
    'd5': 3.0,
}


def fit_head(name, X_train, y_train, seed=RANDOM_STATE):
    model = make_xgb(head_classes[name], seed=seed)
    sample_weight = capped_sample_weight(y_train, cap=WEIGHT_CAPS.get(name, 3.0))
    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def oof_tier_predictions(X_train, train_idx, seed=RANDOM_STATE, n_splits=3):
    y_tier = targets['tier'][train_idx]
    oof_pred = np.zeros(len(train_idx), dtype=int)
    inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for inner_fold, (inner_tr, inner_va) in enumerate(inner.split(X_train, y_tier), start=1):
        model = fit_head('tier', X_train[inner_tr], y_tier[inner_tr], seed=seed + inner_fold)
        oof_pred[inner_va] = model.predict(X_train[inner_va]).astype(int)
    return oof_pred.reshape(-1, 1)


def train_two_stage_heads(X_train, train_idx, seed=RANDOM_STATE, use_oof_tier=True):
    heads = {}
    for name in STAGE1_HEADS:
        y_train = targets[name][train_idx]
        heads[name] = fit_head(name, X_train, y_train, seed=seed)

    tier_feature_train = oof_tier_predictions(X_train, train_idx, seed=seed) if use_oof_tier else heads['tier'].predict(X_train).reshape(-1, 1)
    X_train_aug = np.hstack([X_train, tier_feature_train])

    for name in SCORE_COLS:
        y_train = targets[name][train_idx]
        heads[name] = fit_head(name, X_train_aug, y_train, seed=seed)
    return heads


def predict_head(heads, name, X_base):
    if name in SCORE_COLS:
        tier_pred = heads['tier'].predict(X_base).reshape(-1, 1)
        X_aug = np.hstack([X_base, tier_pred])
        return heads[name].predict(X_aug)
    return heads[name].predict(X_base)


def predict_dim_proba(heads, X_base):
    tier_pred = heads['tier'].predict(X_base).reshape(-1, 1)
    X_aug = np.hstack([X_base, tier_pred])
    return {col: heads[col].predict_proba(X_aug) for col in SCORE_COLS}


def expected_score_from_proba(proba):
    score_values = np.array([CLASS_TO_SCORE[i] for i in range(len(VALID_SCORES))])
    return proba @ score_values


def formula_tier_proba_from_dim_probas(dim_probas):
    expected_dims = pd.DataFrame({col: expected_score_from_proba(dim_probas[col]) for col in SCORE_COLS})
    expected_dims['complexity_score'] = complexity_score_from_dims_frame(expected_dims)
    labels = expected_dims['complexity_score'].apply(tier_from_score).values
    enc = label_encoders['tier'].transform(labels)
    out = np.zeros((len(labels), head_classes['tier']))
    out[np.arange(len(labels)), enc] = 1.0
    return out


def predict_all_dims(heads, X_base):
    tier_pred = heads['tier'].predict(X_base).reshape(-1, 1)
    X_aug = np.hstack([X_base, tier_pred])
    pred_dim_classes = {col: heads[col].predict(X_aug) for col in SCORE_COLS}
    return pd.DataFrame({col: [CLASS_TO_SCORE[int(v)] for v in pred_dim_classes[col]] for col in SCORE_COLS})

## 5-Fold Stratified Cross-Validation

Splits are stratified by `tier`, the main routing target. Identical to Phase 2 v5 — includes blended tier, OOF collection, and confidence diagnostics.

In [10]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_results = {name: {'accuracy': [], 'macro_f1': []} for name in ALL_HEADS}
fold_results['formula_tier'] = {'accuracy': [], 'macro_f1': []}
fold_results['blended_tier'] = {'accuracy': [], 'macro_f1': []}

slice_tier_results = {}
slice_blended_results = {}
oof_true = {name: [] for name in ALL_HEADS + ['formula_tier', 'blended_tier']}
oof_pred = {name: [] for name in ALL_HEADS + ['formula_tier', 'blended_tier']}
oof_tier_proba = []
oof_blended_proba = []

tier_confusions = []
blended_tier_confusions = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, targets['tier']), start=1):
    print(f'\n=== Fold {fold}/{N_SPLITS} ===')

    train_embeddings = embeddings[train_idx]
    val_embeddings = embeddings[val_idx]
    train_hand = hand_df.iloc[train_idx].values
    val_hand = hand_df.iloc[val_idx].values

    fold_pca, fold_scaler, X_train = fit_shared_transformers(train_embeddings, train_hand)
    X_val = transform_shared_features(val_embeddings, val_hand, fold_pca, fold_scaler)

    heads = train_two_stage_heads(X_train, train_idx, seed=RANDOM_STATE + fold, use_oof_tier=True)

    # Evaluate all heads
    for name in ALL_HEADS:
        y_true = targets[name][val_idx]
        y_pred = predict_head(heads, name, X_val)
        fold_results[name]['accuracy'].append(accuracy_score(y_true, y_pred))
        fold_results[name]['macro_f1'].append(f1_score(y_true, y_pred, average='macro', zero_division=0))
        oof_true[name].extend(y_true.tolist())
        oof_pred[name].extend(y_pred.tolist())

    # Direct tier
    direct_tier_pred = predict_head(heads, 'tier', X_val)
    direct_tier_proba = heads['tier'].predict_proba(X_val)
    tier_confusions.append(confusion_matrix(targets['tier'][val_idx], direct_tier_pred, labels=np.arange(head_classes['tier'])))
    oof_tier_proba.append(direct_tier_proba)

    # Formula tier
    dim_probas = predict_dim_proba(heads, X_val)
    formula_proba = formula_tier_proba_from_dim_probas(dim_probas)
    formula_tier_pred = np.argmax(formula_proba, axis=1)
    y_tier_true = targets['tier'][val_idx]
    fold_results['formula_tier']['accuracy'].append(accuracy_score(y_tier_true, formula_tier_pred))
    fold_results['formula_tier']['macro_f1'].append(f1_score(y_tier_true, formula_tier_pred, average='macro', zero_division=0))
    oof_true['formula_tier'].extend(y_tier_true.tolist())
    oof_pred['formula_tier'].extend(formula_tier_pred.tolist())

    # Blended tier
    blended_proba = 0.72 * direct_tier_proba + 0.28 * formula_proba
    blended_pred = np.argmax(blended_proba, axis=1)
    fold_results['blended_tier']['accuracy'].append(accuracy_score(y_tier_true, blended_pred))
    fold_results['blended_tier']['macro_f1'].append(f1_score(y_tier_true, blended_pred, average='macro', zero_division=0))
    oof_true['blended_tier'].extend(y_tier_true.tolist())
    oof_pred['blended_tier'].extend(blended_pred.tolist())
    oof_blended_proba.append(blended_proba)
    blended_tier_confusions.append(confusion_matrix(y_tier_true, blended_pred, labels=np.arange(head_classes['tier'])))

    # Slice diagnostics
    val_df = df.iloc[val_idx]
    cs = val_df['complexity_score'].values
    slices = {
        'all': np.ones(len(val_idx), dtype=bool),
        'boundary_t12': (cs >= 0.35) & (cs <= 0.44),
        'boundary_t23': (cs >= 0.65) & (cs <= 0.74),
        'safe': (cs < 0.30) | ((cs >= 0.45) & (cs <= 0.64)) | (cs >= 0.75),
    }
    for slice_name, mask in slices.items():
        if mask.sum() == 0:
            continue
        slice_tier_results.setdefault(slice_name, []).append(accuracy_score(y_tier_true[mask], direct_tier_pred[mask]))
        slice_blended_results.setdefault(slice_name, []).append(accuracy_score(y_tier_true[mask], blended_pred[mask]))

    fold_blended_acc = accuracy_score(y_tier_true, blended_pred)
    fold_direct_acc = accuracy_score(y_tier_true, direct_tier_pred)
    print(f'Direct tier acc: {fold_direct_acc:.4f}  |  Blended tier acc: {fold_blended_acc:.4f}')


# === Summary ===
print('\n' + '=' * 72)
print('5-Fold CV Results (mean +/- std)')
print('=' * 72)
for name in ALL_HEADS + ['formula_tier', 'blended_tier']:
    acc = np.array(fold_results[name]['accuracy'])
    f1 = np.array(fold_results[name]['macro_f1'])
    print(f'{name:30s} Acc: {acc.mean():.4f} +/- {acc.std():.4f}   F1: {f1.mean():.4f} +/- {f1.std():.4f}')

print('\nDirect tier accuracy by evaluation slice:')
for slice_name, scores in slice_tier_results.items():
    scores = np.array(scores)
    print(f'{slice_name:14s} Acc: {scores.mean():.4f} +/- {scores.std():.4f}')

print('\nBlended tier accuracy by evaluation slice:')
for slice_name, scores in slice_blended_results.items():
    scores = np.array(scores)
    print(f'{slice_name:14s} Acc: {scores.mean():.4f} +/- {scores.std():.4f}')

tier_labels = label_encoders['tier'].classes_
print('\nAggregate direct tier confusion:')
print(pd.DataFrame(np.sum(tier_confusions, axis=0), index=tier_labels, columns=tier_labels))
print('\nAggregate blended tier confusion:')
print(pd.DataFrame(np.sum(blended_tier_confusions, axis=0), index=tier_labels, columns=tier_labels))

print('\nBlended tier classification report:')
print(classification_report(oof_true['blended_tier'], oof_pred['blended_tier'], target_names=tier_labels, zero_division=0))
print('\nD1 classification report:')
print(classification_report(oof_true['d1'], oof_pred['d1'], target_names=[str(v) for v in VALID_SCORES], zero_division=0))
print('\nD2 classification report:')
print(classification_report(oof_true['d2'], oof_pred['d2'], target_names=[str(v) for v in VALID_SCORES], zero_division=0))
print('\nIntent classification report:')
print(classification_report(oof_true['intent'], oof_pred['intent'], target_names=label_encoders['intent'].classes_, zero_division=0))
print('\nTask type classification report:')
print(classification_report(oof_true['task_type'], oof_pred['task_type'], target_names=label_encoders['task_type'].classes_, zero_division=0))

# Confidence diagnostics
oof_blended_proba = np.vstack(oof_blended_proba)
oof_blended_pred = np.array(oof_pred['blended_tier'])
oof_tier_true = np.array(oof_true['blended_tier'])
oof_conf = oof_blended_proba.max(axis=1)
correct = oof_blended_pred == oof_tier_true

print('\nBlended tier confidence diagnostics:')
print('Mean confidence:', round(float(oof_conf.mean()), 4))
for threshold in [0.60, 0.70, 0.80, 0.90]:
    mask = oof_conf >= threshold
    if mask.sum() == 0:
        print(f'Accuracy when confidence >= {threshold:.2f}: no samples')
    else:
        print(
            f'Accuracy when confidence >= {threshold:.2f}:',
            round(float(correct[mask].mean()), 4),
            'coverage:',
            round(float(mask.mean()), 4),
        )


=== Fold 1/5 ===
Direct tier acc: 0.8082  |  Blended tier acc: 0.8062

=== Fold 2/5 ===
Direct tier acc: 0.7955  |  Blended tier acc: 0.8017

=== Fold 3/5 ===
Direct tier acc: 0.8017  |  Blended tier acc: 0.8140

=== Fold 4/5 ===
Direct tier acc: 0.8079  |  Blended tier acc: 0.8140

=== Fold 5/5 ===
Direct tier acc: 0.8533  |  Blended tier acc: 0.8554

5-Fold CV Results (mean +/- std)
tier                           Acc: 0.8133 +/- 0.0205   F1: 0.8225 +/- 0.0202
intent                         Acc: 0.7154 +/- 0.0153   F1: 0.7343 +/- 0.0139
task_type                      Acc: 0.7637 +/- 0.0095   F1: 0.6808 +/- 0.0219
reasoning_chain_detected       Acc: 0.9038 +/- 0.0043   F1: 0.8562 +/- 0.0125
d1                             Acc: 0.6947 +/- 0.0244   F1: 0.7220 +/- 0.0233
d2                             Acc: 0.7398 +/- 0.0249   F1: 0.6881 +/- 0.0244
d3                             Acc: 0.7716 +/- 0.0184   F1: 0.7367 +/- 0.0239
d4                             Acc: 0.8112 +/- 0.0244   F1: 0.713

## Final Full-Dataset Training

After CV, train the final model on all rows so the inference function can be used directly.

In [11]:
final_pca, final_scaler, X_full = fit_full_feature_matrix()
final_train_idx = np.arange(len(df))
heads = train_two_stage_heads(X_full, final_train_idx, seed=RANDOM_STATE, use_oof_tier=True)

print('Final models trained:', list(heads.keys()))

PCA variance explained: 0.5982
Base feature shape: (2421, 110)
Final models trained: ['tier', 'intent', 'task_type', 'reasoning_chain_detected', 'd1', 'd2', 'd3', 'd4', 'd5']


## Rule-Based Research Signals

In [12]:
RESEARCH_SIGNAL_KEYWORDS = {
    'market_research': ['market', 'industry', 'trend', 'tam', 'sam', 'som'],
    'competitive_analysis': ['competitor', 'competitive', 'benchmark', 'rival'],
    'regulatory_compliance': ['regulation', 'regulatory', 'compliance', 'gdpr', 'hipaa', 'sox', 'eu ai act'],
    'security': ['security', 'vulnerability', 'threat', 'risk', 'iam', 'zero trust'],
    'cloud_infrastructure': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'spend', 'budget', 'showback', 'chargeback'],
    'devops': ['ci/cd', 'pipeline', 'deployment', 'sre', 'devops', 'observability'],
    'data_engineering': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai_governance': ['ai governance', 'llm', 'model risk', 'genai', 'guardrail'],
    'system_integration': ['integration', 'api', 'webhook', 'middleware'],
    'supply_chain': ['supply chain', 'inventory', 'procurement', 'logistics'],
    'hr_tech': ['hr', 'employee', 'workforce', 'talent', 'recruiting'],
    'vendor_analysis': ['vendor', 'rfi', 'rfp', 'procurement'],
}


def extract_research_signals(prompt, d4_score):
    if d4_score <= 0:
        return []
    text = str(prompt).lower()
    signals = []
    for signal, keywords in RESEARCH_SIGNAL_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            signals.append(signal)
    return signals if signals else ['external_research']

## Inference Function

In [13]:
def build_features_for_prompts(new_prompts):
    new_prompts_str = [str(prompt) for prompt in new_prompts]
    # BGE prefix for encoding
    prefixed = ['Represent this sentence: ' + p for p in new_prompts_str]
    new_embeddings = embedding_model.encode(
        prefixed,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    new_hand = handcrafted_features(new_prompts_str, phrasing_styles=None)
    return transform_shared_features(new_embeddings, new_hand.values, final_pca, final_scaler)


def max_probability(model, X_part):
    proba = model.predict_proba(X_part)[0]
    return float(np.max(proba))


def predict_prompt(prompt):
    X_one = build_features_for_prompts([prompt])

    direct_tier_proba = heads['tier'].predict_proba(X_one)
    tier_class = int(np.argmax(direct_tier_proba, axis=1)[0])
    direct_tier = label_encoders['tier'].inverse_transform([tier_class])[0]
    intent = label_encoders['intent'].inverse_transform(heads['intent'].predict(X_one))[0]
    task_type = label_encoders['task_type'].inverse_transform(heads['task_type'].predict(X_one))[0]
    reasoning_chain = bool(int(heads['reasoning_chain_detected'].predict(X_one)[0]))

    X_one_aug = np.hstack([X_one, np.array([[tier_class]])])
    predicted_dims = {}
    for col in SCORE_COLS:
        pred_class = int(heads[col].predict(X_one_aug)[0])
        predicted_dims[col] = CLASS_TO_SCORE[pred_class]

    score = (
        predicted_dims['d1'] * 0.35 +
        predicted_dims['d2'] * 0.20 +
        predicted_dims['d3'] * 0.20 +
        predicted_dims['d4'] * 0.15 +
        predicted_dims['d5'] * 0.10
    )
    formula_tier = tier_from_score(score)
    formula_onehot = np.zeros_like(direct_tier_proba)
    formula_onehot[0, label_encoders['tier'].transform([formula_tier])[0]] = 1.0
    blended_proba = 0.72 * direct_tier_proba + 0.28 * formula_onehot
    blended_tier = label_encoders['tier'].inverse_transform([int(np.argmax(blended_proba, axis=1)[0])])[0]

    key_confidences = [
        float(np.max(blended_proba)),
        max_probability(heads['intent'], X_one),
        max_probability(heads['task_type'], X_one),
        max_probability(heads['reasoning_chain_detected'], X_one),
    ]
    confidence = float(np.mean(key_confidences))

    result = {}
    for col in SCORE_COLS:
        result[col] = predicted_dims[col]
        result[f'{col}_label'] = DIMENSION_LABELS[col]

    result.update({
        'complexity_score': round(float(score), 4),
        'tier': blended_tier,
        'direct_tier': direct_tier,
        'formula_tier': formula_tier,
        'intent': intent,
        'task_type': task_type,
        'reasoning_chain_detected': reasoning_chain,
        'research_signals': extract_research_signals(prompt, predicted_dims['d4']),
        'confidence': round(confidence, 4),
        'tier_confidence': round(float(np.max(blended_proba)), 4),
    })
    return result

In [14]:
sample_prompt = 'Design a multi-cloud GenAI governance architecture for a Fortune 500 company, including compliance risks and vendor evaluation criteria.'
print(json.dumps(predict_prompt(sample_prompt), indent=2))

{
  "d1": 1.0,
  "d1_label": "Semantic Complexity",
  "d2": 1.0,
  "d2_label": "Domain Specificity",
  "d3": 1.0,
  "d3_label": "Output Formality",
  "d4": 0.75,
  "d4_label": "Research Dependency",
  "d5": 0.75,
  "d5_label": "Context Requirement",
  "complexity_score": 0.9375,
  "tier": "T3",
  "direct_tier": "T3",
  "formula_tier": "T3",
  "intent": "STRATEGIC",
  "task_type": "reasoning",
  "reasoning_chain_detected": true,
  "research_signals": [
    "regulatory_compliance",
    "security",
    "cloud_infrastructure",
    "ai_governance",
    "vendor_analysis"
  ],
  "confidence": 0.9098,
  "tier_confidence": 0.9422
}


## Demo Export Bundle

In [15]:
import pickle
from google.colab import files

MODEL_BUNDLE_PATH = '/content/phase3_v1_model_bundle.pkl'

model_bundle = {
    'embedding_model_name': EMBEDDING_MODEL_NAME,
    'embedding_query_prefix': 'Represent this sentence: ',
    'pca': final_pca,
    'scaler': final_scaler,
    'heads': heads,
    'label_encoders': label_encoders,
    'class_to_score': CLASS_TO_SCORE,
    'dimension_labels': DIMENSION_LABELS,
    'valid_scores': VALID_SCORES,
    'score_cols': SCORE_COLS,
    'tier_blend_direct_weight': 0.72,
    'tier_blend_formula_weight': 0.28,
}

with open(MODEL_BUNDLE_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)

print(f'Saved model bundle to {MODEL_BUNDLE_PATH}')
files.download(MODEL_BUNDLE_PATH)

Saved model bundle to /content/phase3_v1_model_bundle.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>